In [1]:
# ============================================================
# MD4 — DISTILROBERTA + CLASS-WEIGHTED CROSS-ENTROPY
#
# RECOVERY / REPRODUCTION VERSION
#
# IMPORTANT:
#   - Train:      14,905 examples
#   - Validation: 1,850 examples
#   - Test:       NOT USED
#
# This version:
#   1. reproduces the original MD4 training setup
#   2. saves ALL THREE epoch checkpoints
#   3. evaluates ALL THREE checkpoints
#   4. saves predictions for ALL THREE checkpoints
#   5. saves clean deployable model folders
#   6. creates .tar.gz archives for all three models
#
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import gc
import json
import random
import shutil

from pathlib import Path
from collections import Counter

import numpy as np

import torch
import torch.nn.functional as F

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


# ============================================================
# CONFIG
# ============================================================

SEED = 42


MODEL_NAME = (
    "distilbert/distilroberta-base"
)


MAX_LENGTH = 512


TRAIN_PATH = Path(
    "/kaggle/input/datasets/"
    "malindus/mathdial-move-selection/"
    "train.jsonl"
)


VAL_PATH = Path(
    "/kaggle/input/datasets/"
    "malindus/mathdial-move-selection/"
    "validation.jsonl"
)


# Hugging Face Trainer checkpoints.
OUTPUT_DIR = Path(
    "/kaggle/working/"
    "md4_weighted_distilroberta"
)


# Original-style final epoch outputs.
RESULT_PATH = Path(
    "/kaggle/working/"
    "md4_weighted_distilroberta_validation.json"
)


PREDICTION_PATH = Path(
    "/kaggle/working/"
    "md4_weighted_distilroberta_predictions.jsonl"
)


# New recovery outputs.
RECOVERY_DIR = Path(
    "/kaggle/working/"
    "md4_recovery"
)


RECOVERY_SUMMARY_PATH = (
    RECOVERY_DIR
    /
    "md4_recovery_summary.json"
)


LABELS = [
    "generic",
    "probing",
    "focus",
    "telling",
]


LABEL2ID = {
    label: index
    for index, label
    in enumerate(LABELS)
}


ID2LABEL = {
    index: label
    for label, index
    in LABEL2ID.items()
}


# ============================================================
# ORIGINAL EXPECTED RESULTS
#
# These are NOT used to select the new model.
#
# They are only reference values from the original run.
# ============================================================

ORIGINAL_EXPECTED = {

    1: {
        "accuracy":
            0.41135135135135137,

        "macro_f1":
            0.4190388006095037,
    },

    2: {
        "accuracy":
            0.47675675675675677,

        "macro_f1":
            0.4739965549662406,
    },

    3: {
        "accuracy":
            0.4772972972972973,

        "macro_f1":
            0.4720959134970202,
    },
}


EXPECTED_SHORT = 1507

EXPECTED_LONG = 343


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# GPU CHECK
# ============================================================

print(
    "=" * 78
)

print(
    "MD4 — DISTILROBERTA "
    "+ WEIGHTED CROSS-ENTROPY"
)

print(
    "=" * 78
)


print(
    "\nCUDA available:",
    torch.cuda.is_available()
)


print(
    "GPU count:",
    torch.cuda.device_count()
)


for index in range(
    torch.cuda.device_count()
):

    print(
        f"GPU {index}:",
        torch.cuda.get_device_name(
            index
        )
    )


assert torch.cuda.is_available(), (
    "GPU is required."
)


assert (
    torch.cuda.device_count()
    ==
    2
), (
    "This reproduction expects T4 x2 "
    "to match the original MD4 run."
)


print(
    "\nGPU configuration: PASSED"
)


# ============================================================
# PREPARE OUTPUT DIRECTORIES
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


RECOVERY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# CLEAR PREVIOUS OBJECTS
# ============================================================

for object_name in [

    "model",
    "trainer",
    "model_md2",
    "trainer_md2",
    "model_md3",
    "trainer_md3",

]:

    if object_name in globals():

        del globals()[
            object_name
        ]


gc.collect()

torch.cuda.empty_cache()


# ============================================================
# LOAD JSONL
# ============================================================

def load_jsonl(
    path
):

    rows = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        for line in f:

            line = (
                line.strip()
            )

            if line:

                rows.append(
                    json.loads(
                        line
                    )
                )

    return rows


# ============================================================
# LOAD DATA
# ============================================================

train_rows = load_jsonl(
    TRAIN_PATH
)


val_rows = load_jsonl(
    VAL_PATH
)


assert (
    len(train_rows)
    ==
    14905
)


assert (
    len(val_rows)
    ==
    1850
)


print(
    "\nTrain examples:",
    len(train_rows)
)


print(
    "Validation examples:",
    len(val_rows)
)


# ============================================================
# COMPUTE CLASS WEIGHTS
#
#              total examples
# weight_c = -------------------
#             classes * count_c
#
# ============================================================

train_counts = Counter(

    example[
        "target_move"
    ]

    for example
    in train_rows
)


num_examples = len(
    train_rows
)


num_classes = len(
    LABELS
)


class_weights = np.array([

    num_examples
    /
    (
        num_classes
        *
        train_counts[
            label
        ]
    )

    for label
    in LABELS

], dtype=np.float32)


print(
    "\n"
    + "=" * 78
)

print(
    "CLASS WEIGHTS"
)

print(
    "=" * 78
)


for label, weight in zip(
    LABELS,
    class_weights,
):

    print(

        f"{label:10s}: "

        f"count="
        f"{train_counts[label]:4d} "

        f"weight="
        f"{weight:.6f}"
    )


# ============================================================
# INPUT FORMAT
# ============================================================

def format_history(
    history
):

    if not history:

        return (
            "[No previous conversation]"
        )

    return "\n".join(

        f"{turn['user']}: "
        f"{turn['text']}"

        for turn
        in history
    )


def build_problem_text(
    example
):

    return (

        "Problem:\n"
        f"{example['problem']}"
    )


def build_conversation_text(
    example
):

    return (

        "Conversation:\n"

        f"{format_history(example['history'])}"

        "\n\n"

        "Next teacher pedagogical move:"
    )


# ============================================================
# TOKENIZER
# ============================================================

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME
    )
)


# Preserve recent conversation when truncation happens.
tokenizer.truncation_side = (
    "left"
)


# ============================================================
# PREPARE DATA
# ============================================================

def prepare_rows(
    rows
):

    prepared = []

    for example in rows:

        prepared.append({

            "example_id":
                example[
                    "example_id"
                ],

            "problem_text":
                build_problem_text(
                    example
                ),

            "conversation_text":
                build_conversation_text(
                    example
                ),

            "labels":
                LABEL2ID[
                    example[
                        "target_move"
                    ]
                ],
        })

    return prepared


train_raw = Dataset.from_list(
    prepare_rows(
        train_rows
    )
)


val_raw = Dataset.from_list(
    prepare_rows(
        val_rows
    )
)


# ============================================================
# ORIGINAL VALIDATION TOKEN LENGTHS
#
# IMPORTANT:
# These DistilRoBERTa lengths are the routing lengths used by
# the frozen hybrid.
# ============================================================

val_original_lengths = []


for example in val_rows:

    encoded = tokenizer(

        build_problem_text(
            example
        ),

        build_conversation_text(
            example
        ),

        add_special_tokens=True,

        truncation=False,
    )

    val_original_lengths.append(

        len(
            encoded[
                "input_ids"
            ]
        )
    )


val_original_lengths = np.array(
    val_original_lengths
)


short_mask = (
    val_original_lengths
    <=
    MAX_LENGTH
)


long_mask = (
    val_original_lengths
    >
    MAX_LENGTH
)


print(
    "\nValidation <=512:",
    int(
        short_mask.sum()
    )
)


print(
    "Validation >512:",
    int(
        long_mask.sum()
    )
)


assert (
    int(
        short_mask.sum()
    )
    ==
    EXPECTED_SHORT
)


assert (
    int(
        long_mask.sum()
    )
    ==
    EXPECTED_LONG
)


print(
    "Original routing counts: PASSED"
)


# ============================================================
# TOKENIZE
# ============================================================

def tokenize_batch(
    batch
):

    return tokenizer(

        batch[
            "problem_text"
        ],

        batch[
            "conversation_text"
        ],

        truncation=
            "only_second",

        max_length=
            MAX_LENGTH,
    )


tokenized_train = train_raw.map(

    tokenize_batch,

    batched=True,

    remove_columns=[

        "example_id",
        "problem_text",
        "conversation_text",
    ],
)


tokenized_val = val_raw.map(

    tokenize_batch,

    batched=True,

    remove_columns=[

        "example_id",
        "problem_text",
        "conversation_text",
    ],
)


assert max(

    len(ids)

    for ids
    in tokenized_train[
        "input_ids"
    ]

) <= MAX_LENGTH


assert max(

    len(ids)

    for ids
    in tokenized_val[
        "input_ids"
    ]

) <= MAX_LENGTH


# ============================================================
# MODEL
# ============================================================

model = (

    AutoModelForSequenceClassification
    .from_pretrained(

        MODEL_NAME,

        num_labels=
            len(
                LABELS
            ),

        label2id=
            LABEL2ID,

        id2label=
            ID2LABEL,
    )
)


# ============================================================
# CUSTOM WEIGHTED TRAINER
# ============================================================

class WeightedTrainer(
    Trainer
):

    def __init__(
        self,
        *args,
        class_weights=None,
        **kwargs,
    ):

        super().__init__(
            *args,
            **kwargs,
        )

        self.class_weights_tensor = (
            torch.tensor(
                class_weights,
                dtype=torch.float32,
            )
        )


    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):

        labels = (
            inputs.pop(
                "labels"
            )
        )

        outputs = model(
            **inputs
        )

        logits = (
            outputs.logits
        )

        weights = (

            self.class_weights_tensor
            .to(
                logits.device
            )
        )

        loss = F.cross_entropy(

            logits,

            labels,

            weight=
                weights,
        )

        if return_outputs:

            return (
                loss,
                outputs,
            )

        return loss


# ============================================================
# METRICS
# ============================================================

def compute_metrics(
    eval_prediction
):

    logits, labels = (
        eval_prediction
    )

    predictions = np.argmax(
        logits,
        axis=-1,
    )

    return {

        "accuracy":
            accuracy_score(
                labels,
                predictions,
            ),

        "macro_f1":
            f1_score(

                labels,
                predictions,

                labels=[
                    0,
                    1,
                    2,
                    3,
                ],

                average=
                    "macro",

                zero_division=
                    0,
            ),
    }


# ============================================================
# TRAINING CONFIG
#
# ORIGINAL SETTINGS PRESERVED.
#
# ONLY MAJOR RECOVERY CHANGE:
#
#     save_strategy = "epoch"
#
# so epoch 1, epoch 2 and epoch 3 all survive.
# ============================================================

training_args = TrainingArguments(

    output_dir=
        str(
            OUTPUT_DIR
        ),

    learning_rate=
        2e-5,

    per_device_train_batch_size=
        8,

    per_device_eval_batch_size=
        16,

    num_train_epochs=
        3,

    weight_decay=
        0.01,

    warmup_steps=
        280,

    eval_strategy=
        "epoch",

    logging_strategy=
        "epoch",

    # --------------------------------------------------------
    # RECOVERY CHANGE
    # --------------------------------------------------------

    save_strategy=
        "epoch",

    save_total_limit=
        3,

    # Do NOT automatically load the rerun's best epoch.
    load_best_model_at_end=
        False,

    # --------------------------------------------------------

    fp16=
        True,

    seed=
        SEED,

    data_seed=
        SEED,

    report_to=
        "none",
)


trainer = WeightedTrainer(

    model=
        model,

    args=
        training_args,

    train_dataset=
        tokenized_train,

    eval_dataset=
        tokenized_val,

    processing_class=
        tokenizer,

    data_collator=
        DataCollatorWithPadding(
            tokenizer=
                tokenizer
        ),

    compute_metrics=
        compute_metrics,

    class_weights=
        class_weights,
)


print(
    "\nTrainer GPU count:",
    trainer.args.n_gpu
)


assert (
    trainer.args.n_gpu
    ==
    2
)


# ============================================================
# TRAIN
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "STARTING MD4 TRAINING"
)

print(
    "=" * 78
)


trainer.train()


# ============================================================
# EPOCH HISTORY
# ============================================================

epoch_history = [

    log

    for log
    in trainer.state.log_history

    if (
        "eval_macro_f1"
        in log
    )
]


print(
    "\n"
    + "=" * 78
)

print(
    "MD4 EPOCH VALIDATION HISTORY"
)

print(
    "=" * 78
)


for log in epoch_history:

    print(
        f"\nEpoch: "
        f"{log.get('epoch')}"
    )

    print(
        f"  Validation loss: "
        f"{log.get('eval_loss'):.6f}"
    )

    print(
        f"  Accuracy: "
        f"{log.get('eval_accuracy'):.6f}"
    )

    print(
        f"  Macro-F1: "
        f"{log.get('eval_macro_f1'):.6f}"
    )


assert (
    len(
        epoch_history
    )
    ==
    3
)


# ============================================================
# FIND ALL SAVED CHECKPOINTS
# ============================================================

checkpoint_dirs = sorted(

    [

        path

        for path
        in OUTPUT_DIR.glob(
            "checkpoint-*"
        )

        if path.is_dir()
    ],

    key=lambda path:

        int(
            path.name
            .split("-")[-1]
        ),
)


print(
    "\n"
    + "=" * 78
)

print(
    "SAVED MD4 CHECKPOINTS"
)

print(
    "=" * 78
)


for epoch_number, checkpoint_dir in enumerate(
    checkpoint_dirs,
    start=1,
):

    print(
        f"\nEpoch {epoch_number}:"
    )

    print(
        checkpoint_dir
    )


assert (
    len(
        checkpoint_dirs
    )
    ==
    3
), (
    "Expected exactly 3 checkpoints, "
    f"found {len(checkpoint_dirs)}."
)


# Ensure tokenizer files are present with each model.
for checkpoint_dir in checkpoint_dirs:

    tokenizer.save_pretrained(
        str(
            checkpoint_dir
        )
    )


print(
    "\nAll 3 epoch checkpoints saved: PASSED"
)


# ============================================================
# EVALUATION HELPERS
# ============================================================

def logits_to_probabilities(
    logits
):

    stable_logits = (

        logits
        -
        logits.max(
            axis=1,
            keepdims=True,
        )
    )

    exp_logits = np.exp(
        stable_logits
    )

    return (

        exp_logits
        /
        exp_logits.sum(
            axis=1,
            keepdims=True,
        )
    )


def evaluate_subset(
    true_ids,
    pred_ids,
    mask,
):

    y_true = true_ids[
        mask
    ]

    y_pred = pred_ids[
        mask
    ]


    accuracy = accuracy_score(
        y_true,
        y_pred,
    )


    macro_f1 = f1_score(

        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
        ],

        average=
            "macro",

        zero_division=
            0,
    )


    per_class = f1_score(

        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
        ],

        average=
            None,

        zero_division=
            0,
    )


    matrix = confusion_matrix(

        y_true,
        y_pred,

        labels=[
            0,
            1,
            2,
            3,
        ],
    )


    return {

        "examples":
            int(
                len(
                    y_true
                )
            ),

        "accuracy":
            float(
                accuracy
            ),

        "macro_f1":
            float(
                macro_f1
            ),

        "per_class_f1": {

            label:
                float(
                    score
                )

            for label, score
            in zip(
                LABELS,
                per_class,
            )
        },

        "confusion_matrix":
            matrix.tolist(),
    }


def evaluate_predictions(
    logits,
    true_ids,
):

    pred_ids = np.argmax(
        logits,
        axis=-1,
    )

    all_mask = np.ones(
        len(
            true_ids
        ),
        dtype=bool,
    )

    overall = evaluate_subset(
        true_ids,
        pred_ids,
        all_mask,
    )

    short_result = evaluate_subset(
        true_ids,
        pred_ids,
        short_mask,
    )

    long_result = evaluate_subset(
        true_ids,
        pred_ids,
        long_mask,
    )

    return (
        pred_ids,
        overall,
        short_result,
        long_result,
    )


def save_prediction_jsonl(
    path,
    true_ids,
    pred_ids,
    probabilities,
):

    with path.open(
        "w",
        encoding="utf-8",
    ) as f:

        for index, example in enumerate(
            val_rows
        ):

            record = {

                "example_id":
                    example[
                        "example_id"
                    ],

                "qid":
                    example[
                        "qid"
                    ],

                "history_num_turns":
                    example[
                        "history_num_turns"
                    ],

                "original_token_length":
                    int(
                        val_original_lengths[
                            index
                        ]
                    ),

                "true_move":
                    LABELS[
                        int(
                            true_ids[
                                index
                            ]
                        )
                    ],

                "predicted_move":
                    LABELS[
                        int(
                            pred_ids[
                                index
                            ]
                        )
                    ],

                "confidence":
                    float(
                        probabilities[
                            index
                        ].max()
                    ),

                "probabilities": {

                    label:
                        float(
                            probabilities[
                                index,
                                label_id,
                            ]
                        )

                    for label_id, label
                    in enumerate(
                        LABELS
                    )
                },
            }

            f.write(

                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                +
                "\n"
            )


# ============================================================
# FINAL EPOCH-3 PREDICTIONS
#
# This preserves the original MD4 output filenames.
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "FINAL EPOCH MODEL EVALUATION"
)

print(
    "=" * 78
)


final_prediction_output = (
    trainer.predict(
        tokenized_val
    )
)


final_logits = (
    final_prediction_output.predictions
)


final_true_ids = (
    final_prediction_output.label_ids
)


final_probabilities = (
    logits_to_probabilities(
        final_logits
    )
)


(
    final_pred_ids,
    overall,
    short_result,
    long_result,
) = evaluate_predictions(

    final_logits,
    final_true_ids,
)


print(
    "\nAccuracy:",
    overall[
        "accuracy"
    ]
)


print(
    "Macro-F1:",
    overall[
        "macro_f1"
    ]
)


print(
    "\n<=512 examples:",
    short_result[
        "examples"
    ]
)


print(
    "<=512 Macro-F1:",
    short_result[
        "macro_f1"
    ]
)


print(
    "\n>512 examples:",
    long_result[
        "examples"
    ]
)


print(
    ">512 Macro-F1:",
    long_result[
        "macro_f1"
    ]
)


# ============================================================
# ORIGINAL-STYLE FINAL VALIDATION REPORT
# ============================================================

result = {

    "experiment":
        "MD4_distilroberta_weighted_ce",

    "model":
        MODEL_NAME,

    "max_length":
        MAX_LENGTH,

    "epochs":
        3,

    "loss":
        "inverse_frequency_weighted_cross_entropy",

    "class_counts":
        dict(
            train_counts
        ),

    "class_weights": {

        label:
            float(
                weight
            )

        for label, weight
        in zip(
            LABELS,
            class_weights,
        )
    },

    "overall":
        overall,

    "non_truncated":
        short_result,

    "truncated":
        long_result,

    "epoch_history":
        epoch_history,
}


with RESULT_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        result,
        f,
        indent=2,
    )


save_prediction_jsonl(

    PREDICTION_PATH,

    final_true_ids,

    final_pred_ids,

    final_probabilities,
)


# ============================================================
# PRINT FINAL CLASSIFICATION REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "FINAL EPOCH CLASSIFICATION REPORT"
)

print(
    "=" * 78
)


print(

    classification_report(

        final_true_ids,

        final_pred_ids,

        labels=[
            0,
            1,
            2,
            3,
        ],

        target_names=
            LABELS,

        digits=
            4,

        zero_division=
            0,
    )
)


# ============================================================
# CLEAN FINAL TRAINER FROM GPU BEFORE CHECKPOINT EVALUATION
# ============================================================

del final_prediction_output

del trainer

del model


gc.collect()

torch.cuda.empty_cache()


# ============================================================
# EVALUATE ALL THREE SAVED CHECKPOINTS
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "EVALUATING ALL THREE SAVED MODELS"
)

print(
    "=" * 78
)


evaluation_args = TrainingArguments(

    output_dir=
        str(
            RECOVERY_DIR
            /
            "temporary_evaluation"
        ),

    per_device_eval_batch_size=
        16,

    fp16=
        True,

    seed=
        SEED,

    data_seed=
        SEED,

    report_to=
        "none",
)


checkpoint_results = []


for epoch_number, checkpoint_dir in enumerate(
    checkpoint_dirs,
    start=1,
):

    print(
        "\n"
        + "-" * 78
    )

    print(
        f"EPOCH {epoch_number}"
    )

    print(
        checkpoint_dir
    )

    print(
        "-" * 78
    )


    # --------------------------------------------------------
    # LOAD SAVED MODEL
    # --------------------------------------------------------

    checkpoint_model = (

        AutoModelForSequenceClassification
        .from_pretrained(
            str(
                checkpoint_dir
            )
        )
    )


    checkpoint_trainer = WeightedTrainer(

        model=
            checkpoint_model,

        args=
            evaluation_args,

        eval_dataset=
            tokenized_val,

        processing_class=
            tokenizer,

        data_collator=
            DataCollatorWithPadding(
                tokenizer=
                    tokenizer
            ),

        compute_metrics=
            compute_metrics,

        class_weights=
            class_weights,
    )


    # --------------------------------------------------------
    # PREDICT
    # --------------------------------------------------------

    prediction_output = (
        checkpoint_trainer.predict(
            tokenized_val
        )
    )


    logits = (
        prediction_output.predictions
    )


    true_ids = (
        prediction_output.label_ids
    )


    probabilities = (
        logits_to_probabilities(
            logits
        )
    )


    (
        pred_ids,
        checkpoint_overall,
        checkpoint_short,
        checkpoint_long,
    ) = evaluate_predictions(

        logits,
        true_ids,
    )


    # --------------------------------------------------------
    # COMPARE WITH ORIGINAL RUN
    # --------------------------------------------------------

    expected = (
        ORIGINAL_EXPECTED[
            epoch_number
        ]
    )


    accuracy_difference = abs(

        checkpoint_overall[
            "accuracy"
        ]
        -
        expected[
            "accuracy"
        ]
    )


    macro_f1_difference = abs(

        checkpoint_overall[
            "macro_f1"
        ]
        -
        expected[
            "macro_f1"
        ]
    )


    print(
        "\nObserved accuracy:",
        checkpoint_overall[
            "accuracy"
        ]
    )


    print(
        "Original accuracy:",
        expected[
            "accuracy"
        ]
    )


    print(
        "Accuracy difference:",
        accuracy_difference
    )


    print(
        "\nObserved Macro-F1:",
        checkpoint_overall[
            "macro_f1"
        ]
    )


    print(
        "Original Macro-F1:",
        expected[
            "macro_f1"
        ]
    )


    print(
        "Macro-F1 difference:",
        macro_f1_difference
    )


    print(
        "\n<=512 Macro-F1:",
        checkpoint_short[
            "macro_f1"
        ]
    )


    print(
        ">512 Macro-F1:",
        checkpoint_long[
            "macro_f1"
        ]
    )


    # --------------------------------------------------------
    # SAVE EPOCH PREDICTIONS
    # --------------------------------------------------------

    epoch_prediction_path = (

        RECOVERY_DIR
        /
        (
            f"md4_epoch{epoch_number}_"
            f"{checkpoint_dir.name}_"
            "predictions.jsonl"
        )
    )


    save_prediction_jsonl(

        epoch_prediction_path,

        true_ids,

        pred_ids,

        probabilities,
    )


    # --------------------------------------------------------
    # SAVE EPOCH VALIDATION REPORT
    # --------------------------------------------------------

    epoch_validation_path = (

        RECOVERY_DIR
        /
        (
            f"md4_epoch{epoch_number}_"
            f"{checkpoint_dir.name}_"
            "validation.json"
        )
    )


    epoch_report = {

        "experiment":
            "MD4_recovery_checkpoint",

        "epoch":
            epoch_number,

        "checkpoint":
            checkpoint_dir.name,

        "model":
            MODEL_NAME,

        "max_length":
            MAX_LENGTH,

        "loss":
            "inverse_frequency_weighted_cross_entropy",

        "overall":
            checkpoint_overall,

        "non_truncated":
            checkpoint_short,

        "truncated":
            checkpoint_long,

        "original_expected": {
            "accuracy":
                expected[
                    "accuracy"
                ],

            "macro_f1":
                expected[
                    "macro_f1"
                ],
        },

        "difference_from_original": {
            "accuracy_absolute":
                float(
                    accuracy_difference
                ),

            "macro_f1_absolute":
                float(
                    macro_f1_difference
                ),
        },
    }


    with epoch_validation_path.open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            epoch_report,
            f,
            indent=2,
        )


    # --------------------------------------------------------
    # CREATE CLEAN DEPLOYABLE MODEL DIRECTORY
    #
    # We do not need optimizer state for deployment.
    # --------------------------------------------------------

    deployable_model_dir = (

        RECOVERY_DIR
        /
        (
            f"md4_epoch{epoch_number}_"
            f"{checkpoint_dir.name}_model"
        )
    )


    checkpoint_trainer.save_model(
        str(
            deployable_model_dir
        )
    )


    tokenizer.save_pretrained(
        str(
            deployable_model_dir
        )
    )


    # --------------------------------------------------------
    # ARCHIVE CLEAN MODEL
    # --------------------------------------------------------

    archive_path = shutil.make_archive(

        str(
            deployable_model_dir
        ),

        "gztar",

        root_dir=
            deployable_model_dir,
    )


    print(
        "\nPredictions:"
    )

    print(
        epoch_prediction_path
    )


    print(
        "\nValidation report:"
    )

    print(
        epoch_validation_path
    )


    print(
        "\nDeployable model:"
    )

    print(
        deployable_model_dir
    )


    print(
        "\nArchive:"
    )

    print(
        archive_path
    )


    checkpoint_results.append({

        "epoch":
            epoch_number,

        "checkpoint":
            checkpoint_dir.name,

        "accuracy":
            checkpoint_overall[
                "accuracy"
            ],

        "macro_f1":
            checkpoint_overall[
                "macro_f1"
            ],

        "original_accuracy":
            expected[
                "accuracy"
            ],

        "original_macro_f1":
            expected[
                "macro_f1"
            ],

        "accuracy_absolute_difference":
            float(
                accuracy_difference
            ),

        "macro_f1_absolute_difference":
            float(
                macro_f1_difference
            ),

        "short_examples":
            checkpoint_short[
                "examples"
            ],

        "long_examples":
            checkpoint_long[
                "examples"
            ],

        "prediction_path":
            str(
                epoch_prediction_path
            ),

        "validation_path":
            str(
                epoch_validation_path
            ),

        "model_directory":
            str(
                deployable_model_dir
            ),

        "archive_path":
            str(
                archive_path
            ),
    })


    # --------------------------------------------------------
    # CLEAN GPU BEFORE NEXT CHECKPOINT
    # --------------------------------------------------------

    del prediction_output

    del checkpoint_trainer

    del checkpoint_model


    gc.collect()

    torch.cuda.empty_cache()


# ============================================================
# VALIDATE ALL THREE MODELS WERE SAVED
# ============================================================

assert (
    len(
        checkpoint_results
    )
    ==
    3
)


for checkpoint_result in checkpoint_results:

    assert (
        checkpoint_result[
            "short_examples"
        ]
        ==
        EXPECTED_SHORT
    )

    assert (
        checkpoint_result[
            "long_examples"
        ]
        ==
        EXPECTED_LONG
    )


print(
    "\nAll checkpoint routing counts: PASSED"
)


# ============================================================
# RECOVERY SUMMARY
# ============================================================

recovery_summary = {

    "experiment":
        "MD4_three_epoch_model_recovery",

    "model":
        MODEL_NAME,

    "seed":
        SEED,

    "max_length":
        MAX_LENGTH,

    "training_examples":
        len(
            train_rows
        ),

    "validation_examples":
        len(
            val_rows
        ),

    "test_used":
        False,

    "training_configuration": {

        "learning_rate":
            2e-5,

        "per_device_train_batch_size":
            8,

        "per_device_eval_batch_size":
            16,

        "num_train_epochs":
            3,

        "weight_decay":
            0.01,

        "warmup_steps":
            280,

        "fp16":
            True,

        "seed":
            SEED,

        "data_seed":
            SEED,

        "gpu_count_expected":
            2,

        "save_strategy":
            "epoch",

        "load_best_model_at_end":
            False,
    },

    "class_counts":
        dict(
            train_counts
        ),

    "class_weights": {

        label:
            float(
                weight
            )

        for label, weight
        in zip(
            LABELS,
            class_weights,
        )
    },

    "routing": {
        "le_512":
            int(
                short_mask.sum()
            ),

        "gt_512":
            int(
                long_mask.sum()
            ),
    },

    "original_frozen_behavior_source":
        (
            "Original saved MD4 validation "
            "prediction artifact"
        ),

    "expected_original_frozen_epoch":
        3,

    "checkpoint_results":
        checkpoint_results,

    "important_note":
        (
            "Do not select the recovered frozen model "
            "only from the highest rerun validation Macro-F1. "
            "All three checkpoints must later be compared "
            "against the original saved per-example MD4 "
            "prediction artifact."
        ),
}


with RECOVERY_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        recovery_summary,
        f,
        indent=2,
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "MD4 THREE-EPOCH RECOVERY SUMMARY"
)

print(
    "=" * 78
)


for result in checkpoint_results:

    print(
        f"\nEpoch {result['epoch']} "
        f"({result['checkpoint']})"
    )

    print(
        "  Accuracy:",
        result[
            "accuracy"
        ]
    )

    print(
        "  Original accuracy:",
        result[
            "original_accuracy"
        ]
    )

    print(
        "  Accuracy diff:",
        result[
            "accuracy_absolute_difference"
        ]
    )

    print(
        "  Macro-F1:",
        result[
            "macro_f1"
        ]
    )

    print(
        "  Original Macro-F1:",
        result[
            "original_macro_f1"
        ]
    )

    print(
        "  Macro-F1 diff:",
        result[
            "macro_f1_absolute_difference"
        ]
    )

    print(
        "  Archive:",
        result[
            "archive_path"
        ]
    )


print(
    "\n"
    + "-" * 78
)


print(
    "Recovery summary:"
)

print(
    RECOVERY_SUMMARY_PATH
)


print(
    "\nOriginal-style final validation:"
)

print(
    RESULT_PATH
)


print(
    "\nOriginal-style final predictions:"
)

print(
    PREDICTION_PATH
)


print(
    "\n"
    + "=" * 78
)

print(
    "MD4 RECOVERY RUN COMPLETE"
)

print(
    "ALL THREE EPOCH MODELS SAVED"
)

print(
    "=" * 78
)

MD4 — DISTILROBERTA + WEIGHTED CROSS-ENTROPY

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

GPU configuration: PASSED



Train examples: 14905
Validation examples: 1850

CLASS WEIGHTS
generic   : count=3579 weight=1.041143
probing   : count=3413 weight=1.091781
focus     : count=5443 weight=0.684595
telling   : count=2470 weight=1.508603


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...


Predictions:
/kaggle/working/md4_recovery/md4_epoch2_checkpoint-1864_predictions.jsonl

Validation report:
/kaggle/working/md4_recovery/md4_epoch2_checkpoint-1864_validation.json

Deployable model:
/kaggle/working/md4_recovery/md4_epoch2_checkpoint-1864_model

Archive:
/kaggle/working/md4_recovery/md4_epoch2_checkpoint-1864_model.tar.gz

------------------------------------------------------------------------------
EPOCH 3
/kaggle/working/md4_weighted_distilroberta/checkpoint-2796
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Observed accuracy: 0.4897297297297297
Original accuracy: 0.4772972972972973
Accuracy difference: 0.012432432432432416

Observed Macro-F1: 0.48321092131167
Original Macro-F1: 0.4720959134970202
Macro-F1 difference: 0.011115007814649802

<=512 Macro-F1: 0.4948274167286108
>512 Macro-F1: 0.2150445298133118


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Predictions:
/kaggle/working/md4_recovery/md4_epoch3_checkpoint-2796_predictions.jsonl

Validation report:
/kaggle/working/md4_recovery/md4_epoch3_checkpoint-2796_validation.json

Deployable model:
/kaggle/working/md4_recovery/md4_epoch3_checkpoint-2796_model

Archive:
/kaggle/working/md4_recovery/md4_epoch3_checkpoint-2796_model.tar.gz

All checkpoint routing counts: PASSED

MD4 THREE-EPOCH RECOVERY SUMMARY

Epoch 1 (checkpoint-932)
  Accuracy: 0.40702702702702703
  Original accuracy: 0.41135135135135137
  Accuracy diff: 0.004324324324324336
  Macro-F1: 0.4154491766613886
  Original Macro-F1: 0.4190388006095037
  Macro-F1 diff: 0.003589623948115128
  Archive: /kaggle/working/md4_recovery/md4_epoch1_checkpoint-932_model.tar.gz

Epoch 2 (checkpoint-1864)
  Accuracy: 0.47621621621621624
  Original accuracy: 0.47675675675675677
  Accuracy diff: 0.000540540540540535
  Macro-F1: 0.47640119165172007
  Original Macro-F1: 0.4739965549662406
  Macro-F1 diff: 0.0024046366854794665
  Archive: /k